# 1. Read data from the sales_sample.csv file and analyse to identify problems

EDA - Exploratory Data Analysis

1.1 Define Schema

In [0]:
file_schema = """
    id int,
    name string,
    dop string,
    phone long,
    amount string,
    discount string
"""

1.2 Read Data

In [0]:
sales_sample_raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(file_schema)
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/sales_sample.csv")
)
sales_sample_raw_df.display()

1.3 Describe data

In [0]:
sales_sample_raw_df.describe().display() #To use Exploratory Data Analysis

1.4 List down the problems you want to fix.
-     1. Convert Id from integer to string and rename it as transaction_id
-     2. Rename the name column to customer_name
-     3. Convert the dop date to date format and rename the column to date_of_purchase
-     4. Rename the phone column to customer_phone
-     5. Convert the Amount to Long value and filter the null values and outlier values
-     6. Rename the amount column to purchase_amount
-     7. Convert the discount to double , converting nil and null values to zero. Rename the column to applied_discount.

# 2. Prepare and clear the dataframe using appropriate transformations.

In [0]:
from pyspark.sql.functions import expr, cast

# sales_sample_df_2 = (
#     sales_sample_raw_df.withColumnsRenamed(
#         {
#             "id": "transaction_id",
#             "name": "customer_name",
#             "dop": "date_of_purchase",
#             "phone": "customer_phone",
#             "amount": "purchase_amount",
#             "discount": "applied_discount",
#         }
#     ).withColumns({
#     "id" : expr("cast(id as string) as transaction_id"),
#     "name" : expr("name as customer_name"),
#     "dop" : expr("cast(dop as date) as date_of_purchase"),
#     "phone" : expr("cast(phone as string) as customer_phone"),
#     "amount" : expr("cast(amount as long) as purchase_amount"),
#     "discount" : expr("cast(discount as Double) as applied_discount")   
#     })
# )

sales_sample_df = sales_sample_raw_df.selectExpr(
    "cast(id as string) as transaction_id",
    "cast(name as string) as customer_name",
    "nvl(try_cast(dop as date),to_date(dop,'dd-MM-yyyy')) as date_of_purchase",
    "cast(phone as string) as customer_phone",
    "cast(amount as long) as purchase_amount",
    "nvl(try_cast(discount as double), 0) as applied_discount"
).filter("purchase_amount is not null and purchase_amount < 200000")

sales_sample_df.display()


In [0]:
sales_sample_df.describe("purchase_amount","applied_discount").display()